In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import plotly.express as px
from sklearn.preprocessing import RobustScaler
import seaborn as sns

**Data Storage**


In [ ]:
df=pd.read_csv('creditcard.csv')

Data Hanndling:

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

**Data Cleaning**

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df['Class'].value_counts()

In [ ]:
scalar=RobustScaler()
df['Scaled_Time']=scalar.fit_transform(df['Time'].values.reshape(-1,1))
df['Scaled_Amount']=scalar.fit_transform(df['Amount'].values.reshape(-1,1))
df.drop(['Time','Amount'],axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df_cn = df[df['Class'] ==0]


In [ ]:
df_fraud = df[df['Class'] == 1]
df_fraud

In [ ]:
def remove_outliers_iqr(df, column, threshold=1.5, visualize=True):
    """
    Removes extreme outliers from a specific numeric column using the IQR method.

    Parameters:
    -----------
    df : pd.DataFrame
        The DataFrame containing your data.
    column : str
        The column name where outliers should be removed.
    threshold : float, optional (default=1.5)
        The multiplier for the IQR (higher = fewer outliers removed, lower = more).
    visualize : bool, optional (default=True)
        Whether to show boxplots before and after outlier removal.

    Returns:
    --------
    df_clean : pd.DataFrame
        A new DataFrame with outliers removed for the specified column.
    """

    # Visualize distribution before removal (optional)
    if visualize:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[column], color='skyblue')
        plt.title(f"Before Outlier Removal ({column})")
        plt.show()

    # Calculate Q1 (25th percentile) and Q3 (75th percentile)
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    # Determine upper and lower thresholds
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR

    print(f"\nFeature: {column}")
    print(f"Q1 = {Q1:.4f}, Q3 = {Q3:.4f}, IQR = {IQR:.4f}")
    print(f"Lower bound = {lower_bound:.4f}, Upper bound = {upper_bound:.4f}")

    # Remove outliers
    before = df.shape[0]
    df_clean = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)].copy()
    after = df_clean.shape[0]

    print(f"Removed {before - after} outliers ({100 * (before - after) / before:.2f}%)")

    # Visualize after removal
    if visualize:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df_clean[column], color='lightgreen')
        plt.title(f"After Outlier Removal ({column})")
        plt.show()

    return df_clean

In [ ]:
numerical_columns = df_cn.select_dtypes(include=['number']).columns.tolist()
# Exclude the 'Class' column if it's still present and you don't want to treat it as a numerical feature for outlier removal
if 'Class' in numerical_columns:
    numerical_columns.remove('Class')

for col in numerical_columns:
  df_cn = remove_outliers_iqr(df_cn, col)

Visualization:

In [ ]:
import seaborn as sns

In [ ]:
#Class Distribution

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='Class',data=df,palette=['blue','red'])
plt.title('Class Distribution- Fraud vs NotFraud',fontsize=14)
plt.yscale('log')
plt.xlabel('Class',fontsize=14)
plt.ylabel('Count(log scale)',fontsize=14)
plt.show()

In [ ]:
for i in range(500):
  df_cn = df_cn.sample(n=10000)
  if i == 0:
    df_cn_obs = df_cn.sample(n=1)
  else:
    df_cn_obs = pd.concat([df_cn_obs , df_cn.sample(n=1)], axis=0)

df_cn_obs.shape

In [ ]:
df_merge = pd.concat([df_cn_obs, df_fraud])
df_merge.shape

In [ ]:
X_df = df_merge.drop('Class', axis =1)
y = df_merge['Class']

variances = X_df.var(ddof=1)

# --- 2. Compute correlation with target (use absolute value of Pearson correlation)---
corrs = X_df.apply(lambda col: np.abs(np.corrcoef(col, y)[0, 1]))

# --- 3. Combine variance + correlation ---
# Normalize both to [0,1] before combining (for equal weighting)
var_norm = (variances - variances.min()) / (variances.max() - variances.min())
corr_norm = (corrs - corrs.min()) / (corrs.max() - corrs.min())

# Weighted combined score (tune weights if you want)
combined_score = 0.5 * var_norm + 0.5 * corr_norm

# --- 4. Pick top features ---
top_k = 10  # You can adjust
top_features = combined_score.sort_values(ascending=False).head(top_k).index.tolist()
top_features

In [ ]:
#Box plots to compare the transaction Amount distribution for each class.

In [ ]:
#there is a problem here  need to fix this
plt.figure(figsize=(7,5))
sns.boxplot(x='Class', y='Scaled_Amount', data=df, palette='muted')
plt.title('Transaction Amount Distribution by Class')
plt.xlabel('Class')
plt.ylabel('Scaled Amount')
plt.show()

In [ ]:
#A histogram of the Time feature can help identify if fraud occurs more at specific times.

In [ ]:
plt.figure(figsize=(12,5))

#Not-fraud
plt.subplot(1,2,1)
sns.histplot(df[df['Class'] == 0]['Scaled_Time'], bins=24, color='blue')
plt.title('Non-Fraud Transactions ')
plt.yscale('log')
plt.xlabel('Scaled_Time')
plt.ylabel('Count')

#Fraud
plt.subplot(1,2,2)
sns.histplot(df[df['Class'] == 1]['Scaled_Time'], bins=24, color='red')
plt.title('Fraudulent Transactions')
plt.xlabel('Scaled_Time')
plt.ylabel('Count')

plt.tight_layout()
plt.show()


 Visualization with Plotly:

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

**Class Distribution:**

In [ ]:
class_counts = df['Class'].value_counts().reset_index()
class_counts.columns = ['Class', 'Count']
class_counts['Class'] = class_counts['Class'].astype(str)

# Bar chart
fig1 = px.bar(
    class_counts,
    x='Class',
    y='Count',
    color='Class',
    text='Count',
    title='Class Distribution (Fraud vs Non-Fraud)',
    color_discrete_map={'0': 'blue', '1': 'red'},
    log_y=True
)

fig1.update_layout(
    xaxis=dict(
        tickvals=['0', '1'],
        ticktext=['Non-Fraud (0)', 'Fraud (1)']
    ),
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgray')
)

fig1.show()

In [ ]:
# Pie chart
fig2 = px.pie(
    class_counts,
    names='Class',
    values='Count',
    color='Class',
    color_discrete_map={0: 'blue', 1: 'red'},
    title='Fraud vs Non-Fraud (Percentage)',
    hole=0.5
)
fig2.update_traces(textinfo='percent+label')
fig2.show()

**Box plots to compare the transaction 'Amount' distribution for each class:**

In [ ]:
# Box plot for Scaled_Amount
fig = px.box(
    df,
    x='Class',
    y='Scaled_Amount',
    color='Class',
    title='Transaction Amount Distribution by Class',
    color_discrete_map={0: 'blue', 1: 'red'}
)
fig.show()


**A histogram of the Time feature can help identify if fraud occurs more at specific times:**


In [ ]:
# Histogram of scaled_time
fig = px.histogram(
    df,
    x='Scaled_Time',
    color='Class',
    nbins=50,
    barmode='overlay',
    title='Distribution of Transactions Over Time',
    color_discrete_map={0: 'blue', 1: 'red'},
    opacity=0.7
)
fig.update_layout(
    xaxis_title='Scaled Time',
    yaxis_title='Transaction Count',
    plot_bgcolor='white',
    yaxis_type='log'
)
fig.show()


In [ ]:
fig_notfraud = px.histogram(
    df[df['Class'] == 0],
    x='Scaled_Time',
    nbins=50,
    title='Non-Fraudulent Transaction Time Distribution',
    color_discrete_sequence=['blue']
)
fig_notfraud.update_layout(xaxis_title='Scaled Time', yaxis_title='Count')
fig_notfraud.show()

fig_fraud = px.histogram(
    df[df['Class'] == 1],
    x='Scaled_Time',
    nbins=50,
    title='Fraudulent Transaction Time Distribution',
    color_discrete_sequence=['red']
)
fig_fraud.update_layout(xaxis_title='Scaled Time', yaxis_title='Count')
fig_fraud.show()


**Correlation_Matrices:**

In [ ]:
# Calculating correlation matrix for fraudulent transactions only
corr_matrix = df[df['Class'] == 1].corr()

fig_corr = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu',
    title='Correlation Heatmap (Fraudulent Transactions Only)',
    labels=dict(color='Correlation'),
    aspect='auto'
)

fig_corr.update_layout(
    template='plotly_dark',
    title_font=dict(size=20, color='white'),
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    width=800,
    height=700,
    coloraxis_colorbar=dict(
        title=dict(text='Correlation', font=dict(color='white')),
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=['-1', '-0.5', '0', '0.5', '1'],
        tickfont=dict(color='white')
    )
)

fig_corr.show()


In [ ]:
corr_matrix = df[df['Class'] == 0].corr()

fig_corr = px.imshow(
    corr_matrix,
    color_continuous_scale='RdBu',
    title='Correlation Heatmap (Non-Fraudulent Transactions Only)',
    labels=dict(color='Correlation'),
    aspect='auto'
)

fig_corr.update_layout(
    template='plotly_dark',
    title_font=dict(size=20, color='white'),
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    width=800,
    height=700,
    coloraxis_colorbar=dict(
        title=dict(text='Correlation', font=dict(color='white')),
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=['-1', '-0.5', '0', '0.5', '1'],
        tickfont=dict(color='white')
    )
)

fig_corr.show()

In [ ]:
corr_matrix = df.corr()  # full correlation matrix
corr_with_class = corr_matrix['Class'].sort_values(key=abs, ascending=False)
print(corr_with_class)


In [ ]:
corr_with_class = df.corr()['Class'].sort_values(key=abs, ascending=False).reset_index()
corr_with_class.columns = ['Feature', 'Correlation']

# bar plot
fig = px.bar(
    corr_with_class,
    x='Feature',
    y='Correlation',
    color='Correlation',
    color_continuous_scale='Inferno',
    title='Correlation of Features with Fraud Class (1)',
)


fig.update_layout(
    template='plotly_dark',
    plot_bgcolor='#111111',
    paper_bgcolor='#111111',
    font=dict(color='white', size=13),
    title=dict(x=0.5, font=dict(size=20, color='orange')),
    xaxis_title='Feature',
    yaxis_title='Correlation with Class',
    xaxis=dict(tickangle=45, tickfont=dict(size=11)),
    margin=dict(l=40, r=40, t=80, b=150)         )

fig.update_coloraxes(
    colorbar_title='Correlation',
    cmin=-1,
    cmax=1,
    colorbar_tickcolor='white',
    colorbar_title_side='right'
)

fig.show()


**Violin Plots:**

In [ ]:
# There is problem here
import plotly.graph_objects as go
from plotly.subplots import make_subplots

violin_features = top_features

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=violin_features,
    horizontal_spacing=0.05,
    vertical_spacing=0.15
)

sample_size = 1000  # sample 1000 points from non-fraud for speed

for i, feature in enumerate(violin_features):
    row = i // 5 + 1
    col = i % 5 + 1

    # Violin for Non-Fraud
    non_fraud_y = df[df['Class']==0][feature].sample(sample_size, random_state=42)
    fig.add_trace(
        go.Violin(
            y=non_fraud_y,
            name='Non-Fraud',
            line_color='blue',
            box_visible=True,
            points='all',
            showlegend=(i==0)
        ),
        row=row, col=col
    )

    # Violin for Fraud (all points, few)
    fraud_y = df[df['Class']==1][feature]
    fig.add_trace(
        go.Violin(
            y=fraud_y,
            name='Fraud',
            line_color='red',
            box_visible=True,
            points='all',
            showlegend=False
        ),
        row=row, col=col
    )

fig.update_layout(
    template='plotly_dark',
    title_text='Top 10 Feature Distributions by Class (Violin Plots)',
    title_x=0.5,
    height=700,
    width=1400,
    violinmode='group'
)

fig.show()


 **Pairwise Scatter Plot for top 5 festures most correlated with Class:**

In [ ]:
# Select top 5 features most correlated with Class
top_features = top_features[:4]
top_features.append('Class')  # Include Class for color

# Scatter matrix (pairplot) using Plotly
fig_pair = px.scatter_matrix(
    df[top_features],
    dimensions=top_features[:-1],  # features only
    color='Class',
    color_discrete_map={0: 'blue', 1: 'red'},
    title='Pairplot of Top Features by Class',
    height=800,
    width=800
)

fig_pair.update_layout(template='plotly_dark')
fig_pair.show()


In [ ]:
# I need the scatter plots for all the combinations of top_features

import plotly.graph_objects as go

# Select numeric features only
features = top_features[:4]

for i in range(len(features)):
  for j in range(i+1, len(features)):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_cn[features[i]],
        y = df_cn[features[j]],
        name='Clean',
        opacity=0.6,
        marker_color='green',
        mode = 'markers'
    ))

    fig.add_trace(go.Scatter(
        x=df_fraud[features[i]],
        y = df_fraud[features[j]],
        name='Fraud',
        opacity=0.6,
        marker_color='red',
        mode = 'markers'
    ))

    fig.update_layout(
        barmode='overlay',
        title=f'Feature Distribution: {features[i]} vs {features[j]}',
        xaxis_title=features[i],
        yaxis_title=features[j],
        template='plotly_white'
    )

    fig.show()

**Analysing Outliers:**

In [ ]:
# outliers in Amount
df['Outlier'] = (df['Scaled_Amount'] > 3) | (df['Scaled_Amount'] < -3)

# total outliers
total_outliers = df['Outlier'].sum()
print("Total outliers:", total_outliers)

# how many outliers are fraud
fraud_outliers = df[(df['Outlier']) & (df['Class']==1)].shape[0]
print("Fraud outliers:", fraud_outliers)

# Percentage of outliers that are fraud
percent_fraud_outliers = (fraud_outliers / total_outliers) * 100
print(f"Percentage of outliers that are fraud: {percent_fraud_outliers:.2f}%")


In [ ]:
outlier_only = df[df['Outlier']].groupby('Class').size().reset_index(name='Count')
outlier_only['Class_Label'] = outlier_only['Class'].map({0:'Non-Fraud',1:'Fraud'})

fig = px.bar(
    outlier_only,
    x='Class_Label',
    y='Count',
    color='Class_Label',
    color_discrete_map={'Non-Fraud':'blue','Fraud':'red'},
    text='Count',
    title='Fraud vs Non-Fraud among Outliers'
)
fig.update_layout(template='plotly_dark')
#fig.update_yaxes(type='log', title_text='Count (log scale)')
fig.show()


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# List of features to check for outliers
features = ['Scaled_Amount', 'Scaled_Time'] + [f'V{i}' for i in range(1,29)]

# Threshold for outliers (e.g., |value| > 3)
threshold = 3

# Loop over features
for feature in features:
    df[f'{feature}_Outlier'] = np.abs(df[feature]) > threshold

    # Count fraud vs non-fraud among outliers
    outlier_summary = df[df[f'{feature}_Outlier']].groupby('Class').size().reset_index(name='Count')
    outlier_summary['Class_Label'] = outlier_summary['Class'].map({0:'Non-Fraud', 1:'Fraud'})

    # Skip if no outliers
    if outlier_summary.empty:
        continue

    # Plot
    fig = px.bar(
        outlier_summary,
        x='Class_Label',
        y='Count',
        color='Class_Label',
        color_discrete_map={'Non-Fraud':'blue','Fraud':'red'},
        text='Count',
        title=f'Fraud vs Non-Fraud Outliers for {feature}'
    )
    fig.update_layout(template='plotly_dark')
    fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Features to analyze
features = ['Scaled_Amount', 'Scaled_Time'] + [f'V{i}' for i in range(1,29)]
threshold = 3

# Subplot grid
rows = 5
cols = 6
fig = make_subplots(rows=rows, cols=cols, subplot_titles=features, horizontal_spacing=0.05, vertical_spacing=0.1)

# Loop over features
for i, feature in enumerate(features):
    row = i // cols + 1
    col = i % cols + 1

    # Flag outliers
    df[f'{feature}_Outlier'] = np.abs(df[feature]) > threshold
    outlier_summary = df[df[f'{feature}_Outlier']].groupby('Class').size().reset_index(name='Count')

    # Skip if no outliers
    if outlier_summary.empty:
        continue

    total_outliers = outlier_summary['Count'].sum()
    outlier_summary['Percent'] = (outlier_summary['Count'] / total_outliers * 100).round(2)
    outlier_summary['Class_Label'] = outlier_summary['Class'].map({0:'Non-Fraud', 1:'Fraud'})

    # Add bars for each class
    for _, row_data in outlier_summary.iterrows():
        fig.add_trace(
            go.Bar(
                x=[row_data['Class_Label']],
                y=[row_data['Count']],
                name=row_data['Class_Label'],
                marker_color='blue' if row_data['Class']==0 else 'red',
                showlegend=(i==0),
                customdata=[row_data['Percent']],
                hovertemplate='%{x}<br>Count: %{y}<br>Percentage: %{customdata}%<extra></extra>'
            ),
            row=row, col=col
        )

# Update layout
fig.update_layout(
    template='plotly_dark',
    height=1200, width=1400,
    title_text='Fraud vs Non-Fraud Outliers Across Features',
    title_x=0.5,
    barmode='group'
)

fig.show()


In [ ]:
import seaborn as sns

no_pattern=[23]
pattern=[3,17,14,12]
for i in pattern:
    sns.kdeplot(df[df['Class']==1]["V"+str(i)], label='Fraud', fill=True)
    sns.kdeplot(df[df['Class']==0]['V'+str(i)], label='Non-Fraud', fill=True)
    plt.title('Distribution for Fraud vs Non-Fraud')
    plt.legend()
    plt.show()

In [ ]:
# Create histograms using plotly for df_cn and df_fraud

import plotly.graph_objects as go

# Select numeric features only
features = df_cn_obs.select_dtypes(include=['number']).columns

for feature in features:
    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=df_cn_obs[feature],
        name='Outliers_Normal',
        opacity=0.6,
        marker_color='blue'
    ))

    fig.add_trace(go.Histogram(
        x=df_fraud[feature],
        name='Fraud',
        opacity=0.6,
        marker_color='red'
    ))

    fig.update_layout(
        barmode='overlay',
        title=f'Feature Distribution: {feature}',
        xaxis_title=feature,
        yaxis_title='Count',
        template='plotly_white'
    )

    fig.show()

**Dashboard**

In [ ]:
!pip install dash

In [ ]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px


total_txn = len(df)
fraud_txn = df_fraud.shape[0]
fraud_pct = 100 * fraud_txn / total_txn

# List of features (numeric only, excluding target)
features = [col for col in df.columns if col != 'Class']

app = Dash(__name__)
app.title = "Credit Card Fraud Detection Dashboard"

# ----------------------------------------
# App Layout
# ----------------------------------------
app.layout = html.Div([
    html.H1("💳 Credit Card Fraud Detection Dashboard", style={'textAlign': 'center'}),

    # ----- KPI Summary -----
    html.Div([
        html.Div([
            html.H3("Total Transactions"),
            html.H2(f"{total_txn:,}")
        ], className="card"),

        html.Div([
            html.H3("Fraudulent Transactions"),
            html.H2(f"{fraud_txn:,}")
        ], className="card"),

        html.Div([
            html.H3("Fraud Percentage"),
            html.H2(f"{fraud_pct:.3f}%")
        ], className="card"),
    ], style={'display': 'flex', 'justifyContent': 'space-around', 'margin': '20px'}),

    html.Hr(),

    # ----- Dropdown filters -----
    html.Div([
        html.Div([
            html.Label("Select X-axis Feature:"),
            dcc.Dropdown(features, value='V10', id='x_feature', clearable=False)
        ], style={'width': '48%', 'display': 'inline-block'}),

        html.Div([
            html.Label("Select Y-axis Feature:"),
            dcc.Dropdown(features, value='V14', id='y_feature', clearable=False)
        ], style={'width': '48%', 'display': 'inline-block', 'float': 'right'})
    ], style={'margin': '10px'}),

    html.Hr(),

    # ----- Scatter plot -----
    html.Div([
        html.H3("Scatter Plot (Feature Relationships)"),
        dcc.Graph(id='scatter_plot')
    ]),

    # ----- Histogram -----
    html.Div([
        html.H3("Histogram (Feature Distribution)"),
        html.Label("Select Feature for Histogram:"),
        dcc.Dropdown(features, value='Amount', id='hist_feature', clearable=False),
        dcc.Graph(id='hist_plot')
    ]),

    # ----- Box Plot -----
    html.Div([
        html.H3("Box Plot (Feature vs Class)"),
        html.Label("Select Feature for Box Plot:"),
        dcc.Dropdown(features, value='V12', id='box_feature', clearable=False),
        dcc.Graph(id='box_plot')
    ]),

    # ----- Violin Plot -----
    html.Div([
        html.H3("Violin Plot (Feature vs Class)"),
        html.Label("Select Feature for Violin Plot:"),
        dcc.Dropdown(features, value='V14', id='violin_feature', clearable=False),
        dcc.Graph(id='violin_plot')
    ])
])

# ----------------------------------------
# Callbacks (interactivity)
# ----------------------------------------

# 1️⃣ Scatter plot
@app.callback(
    Output('scatter_plot', 'figure'),
    [Input('x_feature', 'value'),
     Input('y_feature', 'value')]
)
def update_scatter(x_feature, y_feature):
    fig = px.scatter(
        df, x=x_feature, y=y_feature, color='Class',
        color_discrete_map={0: 'blue', 1: 'red'},
        opacity=0.6,
        title=f'{x_feature} vs {y_feature}',
        labels={'Class': 'Transaction Class'}
    )
    fig.update_layout(template='plotly_dark')
    return fig


# 2️⃣ Histogram
@app.callback(
    Output('hist_plot', 'figure'),
    [Input('hist_feature', 'value')]
)
def update_histogram(feature):
    fig = px.histogram(
        df_merge, x=feature, color='Class',
        barmode='overlay', opacity=0.6,
        color_discrete_map={0: 'blue', 1: 'red'},
        title=f'Distribution of {feature}'
    )
    fig.update_layout(template='plotly_white')
    return fig


# 3️⃣ Box plot
@app.callback(
    Output('box_plot', 'figure'),
    [Input('box_feature', 'value')]
)
def update_boxplot(feature):
    fig = px.box(
        df, x='Class', y=feature, color='Class',
        color_discrete_map={0: 'blue', 1: 'red'},
        title=f'Box Plot of {feature} by Class'
    )
    fig.update_layout(template='plotly_white')
    return fig


# 4️⃣ Violin plot
@app.callback(
    Output('violin_plot', 'figure'),
    [Input('violin_feature', 'value')]
)
def update_violin(feature):
    fig = px.violin(
        df, x='Class', y=feature, color='Class',
        box=True, points='all',
        color_discrete_map={0: 'blue', 1: 'red'},
        title=f'Violin Plot of {feature} by Class'
    )
    fig.update_layout(template='plotly_white')
    return fig


if __name__ == '__main__':
    app.run(jupyter_mode='external', port=8050, debug=True)

In [ ]:
print(type(app))